# Long-read DeepVariant PCA (Hail / Terra)

Build global and within-population ancestry PCs from autosomal, biallelic,
LD-pruned DeepVariant SNVs using Hail on Terra.

This notebook **does not modify** `covariates.source_rebuilt.csv.gz`. It writes
versioned PCA outputs for later review and optional integration as `lr_PC*` /
`lr_pop_PC*` fields.

## Spark / Hail cluster (Terra)

Workload class: ~12k joint-callset samples × 22 autosomal VCF shards → QC MT
checkpoint → LD prune → global PCA → several within-population PCAs.
`ld_prune` and `hwe_normalized_pca` are the memory bottlenecks; VCF import is
the I/O / cost bottleneck.

### Recommended first full run

| Role | Machine type | Count | Notes |
| --- | --- | --- | --- |
| Master / driver | `n1-highmem-8` or `n2-highmem-8` | 1 | 8 vCPU / ~52 GB; keep headroom for exports and Spark UI |
| Workers | `n1-highmem-8` or `n2-highmem-8` | **32–50** | Prefer highmem (≥~6.5 GB/core). Standard nodes often OOM on LD prune / PCA |
| Worker disk | SSD | **200–500 GB** each | Shuffle spill + VCF decompress; 100 GB is usually too tight |
| Preemptibles | optional | ≤ half of workers | Fine for import + `variant_qc` / `sample_qc`; risky for long LD-prune / PCA stages |

Start around **40 × `n1-highmem-8`**. Scale toward 50–80 if import or prune is slow; do not “fix” OOMs by adding standard (low-memory) nodes.

### If you already have `qc_for_pca.mt`

Reuse the checkpoint and run PCA-only on **16–32** highmem workers. That is cheaper
and avoids re-importing the 22 chrom shards.

### Spark knobs if you hit memory pressure

- Prefer fewer, fatter executors on highmem nodes, e.g. `spark.executor.cores=4`
  so each executor gets more RAM (set in `hl.init` if the environment allows).
- Raise `LD_MEMORY_PER_CORE` in the config cell (start at `"1g"`; try `"2g"` on OOM).
- Checkpoint after QC (`CHECKPOINT_MT`) before prune/PCA so retries are cheap.
- Tighten AF / call-rate filters only if prune is still too wide after that.

### Practical tips

- Use a **Hail** Cloud Environment (not a plain Python notebook).
- Stop or downsize the cluster after writing `global_pcs.tsv` / `population_pcs.tsv`.
- Within-population PCA reuses the QC MT but still re-prunes per group; budget
  wall time for ~6–7 large ancestry subsets after the global run.

## Analysis design

- **Global PCs** are the primary association covariates.
- **Within-population PCs** are supplemental. Subsets use provisional
  `ancestry_pred_other` labels from the covariates table.
- Prefer an existing MatrixTable / VariantDataset if you already imported the
  joint DeepVariant callset. Otherwise import the Terra `GL_INTERVAL_set`
  table columns **`VCF`** + **`VCF_idx`** (production joint-calling
  `chr*.g.vcf.bgz` shards). Keep autosomes only (`chr1`–`chr22`); drop
  `chrM` / `chrX` / `chrY`. Do **not** use FastFilter `output_vcf` unless you
  intentionally want that filtered callset. Intersect samples with
  `final_releasable_v9`.
- Filters: autosomes, biallelic SNVs, PASS-only variants, AF in `[0.01, 0.99]`,
  call-rate thresholds, and LD pruning before `hwe_normalized_pca`.

## Outputs

Under `gs://<workspace-bucket>/pca/deepvariant_<run_label>/`:

- `global_pcs.tsv` with `lr_PC1`–`lr_PC32` (parity with AoU short-read `PC1`–`PC32`)
- `population_pcs.tsv` with `lr_pop_PC1`–`lr_pop_PC32`
- eigenvalues / loadings / run metadata
- optional QC MatrixTable checkpoint


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Bootstrap scripts/ from $WORKSPACE_BUCKET/scripts/ when not on the VM.
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from __future__ import annotations

from terra_notebook import init_notebook

SCRIPTS = init_notebook("workspace_paths.py")
from workspace_paths import data_root

import json
import os

import pandas as pd

try:
    display
except NameError:
    def display(value):
        print(value)

ROOT = data_root()
WORKSPACE_BUCKET = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
RUN_LABEL = os.environ.get("PCA_RUN_LABEL", "deepvariant_lr_v1")

if WORKSPACE_BUCKET:
    _bucket = (
        WORKSPACE_BUCKET
        if WORKSPACE_BUCKET.startswith("gs://")
        else f"gs://{WORKSPACE_BUCKET}"
    )
    COV_URI = os.environ.get(
        "PCA_COV_URI", f"{_bucket}/covariates/covariates.source_rebuilt.csv.gz"
    )
    CHROM_MANIFEST_URI = os.environ.get(
        "PCA_CHROM_MANIFEST_URI", f"{_bucket}/manifests/GL_INTERVAL_set.tsv"
    )
    OUT_DIR = os.environ.get("PCA_OUT_DIR", f"{_bucket}/pca/{RUN_LABEL}")
    EXISTING_MT = os.environ.get("PCA_EXISTING_MT", f"{_bucket}/mt/deepvariant_joint.mt")
    EXISTING_VDS = os.environ.get("PCA_EXISTING_VDS", f"{_bucket}/vds/deepvariant_joint.vds")
else:
    COV_URI = os.environ.get("PCA_COV_URI", str(ROOT / "covariates.source_rebuilt.csv.gz"))
    CHROM_MANIFEST_URI = os.environ.get(
        "PCA_CHROM_MANIFEST_URI", str(ROOT / "manifests" / "GL_INTERVAL_set.tsv")
    )
    OUT_DIR = os.environ.get("PCA_OUT_DIR", str(ROOT / "pca" / RUN_LABEL))
    EXISTING_MT = os.environ.get("PCA_EXISTING_MT", "")
    EXISTING_VDS = os.environ.get("PCA_EXISTING_VDS", "")

INPUT_MODE = os.environ.get("PCA_INPUT_MODE", "vcf_list")
CHROM_ID_COLUMN = "entity:GL_INTERVAL_set_id"
CHROM_URI_COLUMN = "VCF"
CHROM_IDX_COLUMN = "VCF_idx"
VCF_URIS: list[str] | None = None
AUTOSOMES_ONLY = True

N_PCS = int(os.environ.get("PCA_N_PCS", "32"))
MIN_AF = 0.01
MAX_AF = 0.99
MIN_VARIANT_CALL_RATE = 0.98
MIN_SAMPLE_CALL_RATE = 0.98
LD_R2 = 0.1
LD_BP_WINDOW = 500_000
LD_MEMORY_PER_CORE = os.environ.get("PCA_LD_MEMORY_PER_CORE", "1g")
PASS_ONLY = True
MIN_POPULATION_N = 100
POPULATION_LABEL = "ancestry_pred_other"
ALLOW_PARTIAL_VCF_OVERLAP = True
RUN_PIPELINE = os.environ.get("PCA_RUN_PIPELINE", "").lower() in {"1", "true", "yes"}

CHECKPOINT_MT = f"{OUT_DIR}/checkpoints/qc_for_pca.mt"
GLOBAL_PCS_TSV = f"{OUT_DIR}/global_pcs.tsv"
POP_PCS_TSV = f"{OUT_DIR}/population_pcs.tsv"
METADATA_JSON = f"{OUT_DIR}/run_metadata.json"

print("ROOT:", ROOT)
print("WORKSPACE_BUCKET:", WORKSPACE_BUCKET or "(local)")
print("OUT_DIR:", OUT_DIR)
print("INPUT_MODE:", INPUT_MODE)
print("RUN_PIPELINE:", RUN_PIPELINE)


In [ ]:
import hail as hl

# On Terra, use a Hail Cloud Environment with highmem workers (see markdown above).
# If LD prune / PCA OOMs, prefer fewer cores per executor so each gets more RAM:
#   hl.init(..., spark_conf={"spark.executor.cores": "4"})
if RUN_PIPELINE:
    hl.init(default_reference="GRCh38", idempotent=True)
    print("Hail version:", hl.version())
else:
    print("Dry run: Hail is not initialized until RUN_PIPELINE=True")


## 1. Load covariates and define the PCA cohort

Use final-releasable long-read samples. Short-read ancestry labels are used only
to define provisional within-population subsets.


In [ ]:
def read_covariates(uri: str) -> pd.DataFrame:
    # pandas can read local paths and many remote URIs; on Terra prefer a copied
    # workspace object if direct HTTPS/GCS access is awkward from the notebook VM.
    return pd.read_csv(uri, dtype={"research_id": str}, low_memory=False)


cov = read_covariates(COV_URI) if RUN_PIPELINE else pd.read_csv(
    ROOT / "covariates.source_rebuilt.csv.gz",
    dtype={"research_id": str},
    low_memory=False,
)
assert cov["research_id"].is_unique
pca_cohort = cov.loc[cov["final_releasable_v9"]].copy()
print(f"Final-releasable candidates: {len(pca_cohort):,}")

population_counts = (
    pca_cohort[POPULATION_LABEL]
    .fillna("NA")
    .value_counts(dropna=False)
    .rename_axis(POPULATION_LABEL)
    .reset_index(name="n")
)
display(population_counts)

# Hail sample annotation table
sample_ann = pca_cohort[[
    "research_id",
    "lr_phase",
    "final_releasable_v9",
    "technology",
    "platform",
    "GC",
    "ancestry_pred",
    "ancestry_pred_other",
    "population",
]].rename(columns={"research_id": "s"})


## 2. Resolve DeepVariant autosomal inputs

Read the Terra `GL_INTERVAL_set` export. Default columns are `VCF` and
`VCF_idx` from `production_joint_calling/outputs/Chromosomes/`. Autosomes only.
Hail discovers the `.tbi` beside each `VCF` path (`VCF_idx` is validated).


In [ ]:
def load_gl_interval_manifest(manifest_uri: str) -> pd.DataFrame:
    manifest = pd.read_csv(manifest_uri, sep="\t", dtype=str)
    id_col = CHROM_ID_COLUMN
    if id_col not in manifest.columns:
        candidates = [c for c in manifest.columns if c.startswith("entity:") or c.endswith("_id")]
        assert candidates, list(manifest.columns)
        id_col = candidates[0]
    for col in (CHROM_URI_COLUMN, CHROM_IDX_COLUMN):
        assert col in manifest.columns, f"missing {col}; columns={list(manifest.columns)}"
    out = manifest.rename(columns={id_col: "interval_id"}).copy()
    out["interval_id"] = out["interval_id"].astype(str).str.strip()
    out[CHROM_URI_COLUMN] = out[CHROM_URI_COLUMN].astype(str).str.strip()
    out[CHROM_IDX_COLUMN] = out[CHROM_IDX_COLUMN].astype(str).str.strip()
    out = out.replace({"": pd.NA, "nan": pd.NA})
    out = out.dropna(subset=[CHROM_URI_COLUMN, CHROM_IDX_COLUMN])
    if AUTOSOMES_ONLY:
        autosomes = {f"chr{i}" for i in range(1, 23)}
        out = out.loc[out["interval_id"].isin(autosomes)].copy()
    assert not out.empty, f"No autosomal VCF rows in {manifest_uri}"
    assert out["interval_id"].is_unique
    return out.reset_index(drop=True)


def natural_chrom_key(interval_id: str) -> tuple[int, str]:
    token = str(interval_id).lower()
    for chrom in range(1, 23):
        if token == f"chr{chrom}":
            return chrom, token
    return 999, token


if VCF_URIS is not None:
    chrom_manifest = pd.DataFrame({"interval_id": [f"uri{i}" for i in range(len(VCF_URIS))], CHROM_URI_COLUMN: list(VCF_URIS)})
    vcf_uris = list(VCF_URIS)
elif INPUT_MODE == "vcf_list":
    manifest_uri = CHROM_MANIFEST_URI
    if not RUN_PIPELINE:
        candidates = [
            ROOT / "manifests" / "GL_INTERVAL_set.tsv",
            ROOT / "tractor_mix" / "manifests" / "GL_INTERVAL_set.tsv",
            Path.cwd() / "tractor_mix" / "manifests" / "GL_INTERVAL_set.tsv",
        ]
        for local_manifest in candidates:
            if local_manifest.exists():
                manifest_uri = str(local_manifest)
                break
    chrom_manifest = load_gl_interval_manifest(manifest_uri)
    chrom_manifest = chrom_manifest.sort_values("interval_id", key=lambda s: s.map(natural_chrom_key))
    vcf_uris = chrom_manifest[CHROM_URI_COLUMN].tolist()
    print(f"Manifest: {manifest_uri}")
    print(f"Intervals: {', '.join(chrom_manifest['interval_id'])}")
    # VCF_idx should sit beside each VCF; Hail uses the sibling .tbi automatically.
    mismatched = chrom_manifest.loc[
        ~chrom_manifest[CHROM_IDX_COLUMN].str.endswith(".tbi")
        | ~chrom_manifest.apply(
            lambda r: r[CHROM_IDX_COLUMN].startswith(r[CHROM_URI_COLUMN]),
            axis=1,
        )
    ]
    if len(mismatched):
        print(f"WARNING: {len(mismatched)} VCF/VCF_idx pairs look mismatched")
        display(mismatched[["interval_id", CHROM_URI_COLUMN, CHROM_IDX_COLUMN]])
else:
    chrom_manifest = pd.DataFrame()
    vcf_uris = []

print(f"Resolved {len(vcf_uris)} autosomal shards")
for uri in vcf_uris[:5]:
    print(" ", uri)
if len(vcf_uris) > 5:
    print("  ...")


## 3. Import / checkpoint a QC MatrixTable for PCA

If contig names are already `chr1`…`chr22`, leave `FIND_REPLACE=None`.
If they are bare `1`…`22`, set `FIND_REPLACE` appropriately or rely on
Hail's contig recoding for GRCh38.


In [ ]:
FIND_REPLACE = None  # e.g. ("^([0-9]+)$", "chr\\1") only if needed


def import_genotype_mt():
    if INPUT_MODE == "mt":
        return hl.read_matrix_table(EXISTING_MT)
    if INPUT_MODE == "vds":
        vds = hl.vds.read_vds(EXISTING_VDS)
        return hl.vds.to_dense_mt(vds)
    if INPUT_MODE == "vcf_list":
        assert vcf_uris, "Set VCF_URIS or CHROM_MANIFEST_URI for INPUT_MODE='vcf_list'"
        # Production chrom shards are named *.g.vcf.bgz but are joint-called
        # multi-sample VCFs with sibling .tbi paths in VCF_idx.
        kwargs = dict(
            path=vcf_uris,
            force_bgz=True,
            reference_genome="GRCh38",
            array_elements_required=False,
        )
        if FIND_REPLACE is not None:
            kwargs["find_replace"] = FIND_REPLACE
        return hl.import_vcf(**kwargs)
    raise ValueError(f"Unsupported INPUT_MODE={INPUT_MODE!r}")


def annotate_samples(mt, sample_df: pd.DataFrame):
    ht = hl.Table.from_pandas(sample_df).key_by("s")
    return mt.annotate_cols(**ht[mt.s])


def filter_for_pca(mt):
    mt = mt.filter_rows(mt.locus.in_autosome())
    mt = mt.filter_rows(hl.len(mt.alleles) == 2)
    mt = mt.filter_rows(hl.is_snp(mt.alleles[0], mt.alleles[1]))
    if PASS_ONLY:
        mt = mt.filter_rows(hl.is_missing(mt.filters) | (hl.len(mt.filters) == 0))
    mt = mt.filter_cols(mt.final_releasable_v9)
    mt = hl.variant_qc(mt)
    mt = hl.sample_qc(mt)
    mt = mt.filter_rows(
        (mt.variant_qc.AF[1] >= MIN_AF)
        & (mt.variant_qc.AF[1] <= MAX_AF)
        & (mt.variant_qc.call_rate >= MIN_VARIANT_CALL_RATE)
    )
    mt = mt.filter_cols(mt.sample_qc.call_rate >= MIN_SAMPLE_CALL_RATE)
    return mt.checkpoint(CHECKPOINT_MT, overwrite=True)


if RUN_PIPELINE:
    mt = import_genotype_mt()
    print("Imported MT:", mt.count())

    mt_samples = set(mt.s.collect())
    requested = set(sample_ann["s"])
    overlap = sorted(requested & mt_samples)
    missing = sorted(requested - mt_samples)
    print(f"VCF samples: {len(mt_samples):,}")
    print(f"Requested releasable: {len(requested):,}")
    print(f"Overlap: {len(overlap):,}; missing from VCF: {len(missing):,}")
    if missing and not ALLOW_PARTIAL_VCF_OVERLAP:
        raise AssertionError(
            "Releasable covariate IDs missing from genotype matrix. "
            f"Examples: {missing[:10]}"
        )
    sample_ann_use = sample_ann.loc[sample_ann["s"].isin(overlap)].copy()
    mt = annotate_samples(mt, sample_ann_use)
    mt = filter_for_pca(mt)
    n_variants, n_samples = mt.count()
    print(f"PCA QC MT: {n_variants:,} variants x {n_samples:,} samples")
else:
    print("Dry run: genotype import / QC skipped")


## 4. Global LD-pruned PCA

This is the primary ancestry covariate set for association models.


In [ ]:
def run_pca(mt, *, n_pcs: int):
    pruned = hl.ld_prune(
        mt.GT,
        r2=LD_R2,
        bp_window_size=LD_BP_WINDOW,
        memory_per_core=LD_MEMORY_PER_CORE,
    )
    mt_pruned = mt.filter_rows(hl.is_defined(pruned[mt.row_key]))
    eigenvalues, scores, loadings = hl.hwe_normalized_pca(
        mt_pruned.GT,
        k=n_pcs,
        compute_loadings=True,
    )
    return scores, loadings, eigenvalues


def scores_to_dataframe(scores, *, prefix: str) -> pd.DataFrame:
    pdf = scores.to_pandas()
    pc_cols = [f"{prefix}{i}" for i in range(1, N_PCS + 1)]
    expanded = pd.DataFrame(pdf["scores"].tolist(), columns=pc_cols)
    out = pd.concat([pdf[["s"]].rename(columns={"s": "research_id"}), expanded], axis=1)
    assert out["research_id"].is_unique
    return out


if RUN_PIPELINE:
    global_scores, global_loadings, global_eigenvalues = run_pca(mt, n_pcs=N_PCS)
    global_pcs = scores_to_dataframe(global_scores, prefix="lr_PC")
    hl.Table.from_pandas(global_pcs).export(GLOBAL_PCS_TSV)
    global_loadings.write(f"{OUT_DIR}/global_loadings.ht", overwrite=True)
    with hl.hadoop_open(f"{OUT_DIR}/global_eigenvalues.txt", "w") as handle:
        for value in global_eigenvalues:
            handle.write(f"{value}\n")
    display(global_pcs.head())
    print("wrote", GLOBAL_PCS_TSV)
else:
    print("Dry run: global PCA skipped")


## 5. Within-population PCA

Supplemental only. Each sufficiently large `ancestry_pred_other` group is
filtered and LD-pruned independently.


In [ ]:
if RUN_PIPELINE:
    pop_tables = []
    pop_values = mt.aggregate_cols(hl.agg.counter(mt[POPULATION_LABEL]))
    for population, n in sorted(pop_values.items(), key=lambda item: (str(item[0]), -item[1])):
        if population is None or n < MIN_POPULATION_N:
            print(f"skip {population!r}: n={n}")
            continue
        print(f"PCA for {population}: n={n}")
        mt_pop = mt.filter_cols(mt[POPULATION_LABEL] == population)
        mt_pop = hl.variant_qc(mt_pop)
        mt_pop = mt_pop.filter_rows(
            (mt_pop.variant_qc.AF[1] >= MIN_AF)
            & (mt_pop.variant_qc.AF[1] <= MAX_AF)
            & (mt_pop.variant_qc.call_rate >= MIN_VARIANT_CALL_RATE)
        )
        scores, loadings, eigenvalues = run_pca(mt_pop, n_pcs=N_PCS)
        pdf = scores_to_dataframe(scores, prefix="lr_pop_PC")
        pdf.insert(1, "population", population)
        pop_tables.append(pdf)
        loadings.write(f"{OUT_DIR}/populations/{population}/loadings.ht", overwrite=True)
        with hl.hadoop_open(f"{OUT_DIR}/populations/{population}/eigenvalues.txt", "w") as handle:
            for value in eigenvalues:
                handle.write(f"{value}\n")

    population_pcs = pd.concat(pop_tables, ignore_index=True)
    assert population_pcs["research_id"].is_unique
    hl.Table.from_pandas(population_pcs).export(POP_PCS_TSV)
    display(population_pcs.groupby("population").size().rename("n").reset_index())
    print("wrote", POP_PCS_TSV)
else:
    print("Dry run: within-population PCA skipped")


## 6. Provenance and next steps

Review PC plots and association inflation before replacing short-read PCs.
Recommended integration later: add `lr_PC1`–`lr_PC32` (and optional
`lr_pop_PC*`) beside the existing AoU short-read PCs rather than overwriting them.


In [ ]:
metadata = {
    "run_label": RUN_LABEL,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "engine": "hail",
    "input_mode": INPUT_MODE,
    "paths": {
        "workspace_bucket": WORKSPACE_BUCKET,
        "covariates": COV_URI,
        "existing_mt": EXISTING_MT,
        "existing_vds": EXISTING_VDS,
        "chrom_manifest": CHROM_MANIFEST_URI,
        "chrom_id_column": CHROM_ID_COLUMN,
        "chrom_uri_column": CHROM_URI_COLUMN,
        "chrom_idx_column": CHROM_IDX_COLUMN,
        "autosomes_only": AUTOSOMES_ONLY,
        "out_dir": OUT_DIR,
        "checkpoint_mt": CHECKPOINT_MT,
        "global_pcs": GLOBAL_PCS_TSV,
        "population_pcs": POP_PCS_TSV,
    },
    "cohort": {
        "global_definition": "final_releasable_v9 == True, intersected with genotype samples",
        "within_population_label_source": POPULATION_LABEL,
        "min_population_n": MIN_POPULATION_N,
        "allow_partial_vcf_overlap": ALLOW_PARTIAL_VCF_OVERLAP,
    },
    "filters": {
        "autosomal_biallelic_snps": True,
        "pass_only": PASS_ONLY,
        "min_af": MIN_AF,
        "max_af": MAX_AF,
        "min_variant_call_rate": MIN_VARIANT_CALL_RATE,
        "min_sample_call_rate": MIN_SAMPLE_CALL_RATE,
        "ld_r2": LD_R2,
        "ld_bp_window": LD_BP_WINDOW,
        "n_pcs": N_PCS,
    },
    "status": "completed" if RUN_PIPELINE else "dry_run",
}

if RUN_PIPELINE:
    with hl.hadoop_open(METADATA_JSON, "w") as handle:
        handle.write(json.dumps(metadata, indent=2) + "\n")
else:
    local_out = ROOT / "pca" / RUN_LABEL
    local_out.mkdir(parents=True, exist_ok=True)
    (local_out / "run_metadata.hail_dry_run.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )
    print("wrote", local_out / "run_metadata.hail_dry_run.json")

print(json.dumps(metadata, indent=2))
